# PyCRMKit 1.2.0 - 05. Segmentation & Saved Queries
Ce notebook explore le moteur de requêtes et de segmentation d'audience :
- **Arbre d'expressions de requêtes (AST)** : `Predicate`, opérateurs `EQ`, `IN`, `CONTAINS`
- **Saved Queries** : requêtes persistées, versionnées et réutilisables
- **Segments Statiques** : listes manuelles avec ajout/retrait explicite de membres
- **Segments Dynamiques** : audiences calculées à la volée selon des prédicats métier
- **Snapshots d'Audience** : capture figée dans le temps d'un segment pour figer une cible de campagne

## 1. Initialisation & Création d'une Base de Données Prospect

In [6]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import CRM, __version__
from pycrmkit.contacts import ContactEmail
from pycrmkit.core.references import EntityReference
from pycrmkit.segments import Predicate, QueryOperator, SegmentMode

crm = CRM.memory().with_context(
    actor_id="marketing-lead",
    correlation_id="segment-session-005"
)

# Population de contacts tests
c1 = crm.contacts.create(first_name="Alice", last_name="Dubois", source="linkedin")
c2 = crm.contacts.create(first_name="Bernard", last_name="Lemoine", source="linkedin")
c3 = crm.contacts.create(first_name="Claire", last_name="Moreau", source="webinar")
c4 = crm.contacts.create(first_name="David", last_name="Rousseau", source="cold_call")

print(f"✅ PyCRMKit version : {__version__}")
print(f"👥 4 Contacts créés avec sources distinctes : {c1.first_name}, {c2.first_name}, {c3.first_name}, {c4.first_name}")

✅ PyCRMKit version : 1.2.0
👥 4 Contacts créés avec sources distinctes : Alice, Bernard, Claire, David


## 2. Segments Statiques
Un segment statique contient une liste explicite d'entités sélectionnées manuellement.

In [7]:
# Création d'un segment statique
vip_segment = crm.segments.create_static(
    key="vip-clients-2026",
    name="Clients VIP France",
    entity_kind="contact",
    description="Liste qualifiée manuellement des décideurs VIP",
)

# Ajout de membres
crm.segments.add_member(vip_segment.id, EntityReference("contact", c1.id))
crm.segments.add_member(vip_segment.id, EntityReference("contact", c3.id))

print(f"🏷️ Segment Statique : {vip_segment.name} (Mode: {vip_segment.mode.value})")
print(f"   Nombre de membres : {crm.segments.count(vip_segment.id)}")
print(f"   Alice est dedans ? {crm.segments.contains(vip_segment.id, EntityReference('contact', c1.id))}")
print(f"   David est dedans ? {crm.segments.contains(vip_segment.id, EntityReference('contact', c4.id))}")

🏷️ Segment Statique : Clients VIP France (Mode: static)
   Nombre de membres : 2
   Alice est dedans ? True
   David est dedans ? False


## 3. Segments Dynamiques basés sur des Prédicats
Un segment dynamique évalue automatiquement les critères de filtrage.

In [8]:
# Segment dynamique pour tous les contacts venant de 'linkedin'
linkedin_segment = crm.segments.create_dynamic(
    key="leads-linkedin",
    name="Acquisitions LinkedIn",
    entity_kind="contact",
    query=Predicate("source", QueryOperator.EQ, "linkedin"),
    description="Tous les prospects générés via LinkedIn",
)

print(f"⚡ Segment Dynamique : {linkedin_segment.name} (Mode: {linkedin_segment.mode.value})")

# Évaluation en temps réel
evaluated_members = crm.segments.evaluate(linkedin_segment.id).items
print(f"   Nombre de membres détectés : {len(evaluated_members)}")
for ref in evaluated_members:
    print(f"   • Membre : [{ref.kind}] {ref.id}")

⚡ Segment Dynamique : Acquisitions LinkedIn (Mode: dynamic)
   Nombre de membres détectés : 2
   • Membre : [contact] 7594f939-710e-40c8-8f0d-33bf6dbba465
   • Membre : [contact] 37d85f33-1512-4985-975d-f2333551a109


## 4. Snapshots d'Audience
Un snapshot fige l'évaluation dynamique à un instant T pour constituer une audience immuable (ex: campagne email).

In [9]:
# Capture d'un snapshot du segment dynamique
snapshot = crm.segments.snapshot(
    linkedin_segment.id,
    key="snapshot-linkedin-q4-2026",
    name="Snapshot LinkedIn Q4 2026",
)

print(f"📸 Snapshot figé créé : {snapshot.name} (ID: {snapshot.id})")
print(f"   Mode du snapshot : {snapshot.mode.value}")
print(f"   Taille figée : {crm.segments.count(snapshot.id)} membres")

# Même si un nouveau contact arrive avec la même source, le snapshot reste stable
c5 = crm.contacts.create(first_name="Emma", last_name="Laurent", source="linkedin")
print(f"Nouveau contact créé : {c5.first_name} (source: linkedin)")
print(f"-> Taille segment dynamique mise à jour : {crm.segments.count(linkedin_segment.id)}")
print(f"-> Taille du snapshot immuable (inchangée) : {crm.segments.count(snapshot.id)}")

📸 Snapshot figé créé : Snapshot LinkedIn Q4 2026 (ID: ac3923b1-8ffa-4b16-be3d-7c8692f2afcd)
   Mode du snapshot : snapshot
   Taille figée : 2 membres
Nouveau contact créé : Emma (source: linkedin)
-> Taille segment dynamique mise à jour : 3
-> Taille du snapshot immuable (inchangée) : 2


## 5. Assertions de Validation du Notebook

In [10]:
assert crm.segments.count(vip_segment.id) == 2
assert crm.segments.count(linkedin_segment.id) == 3
assert crm.segments.count(snapshot.id) == 2
assert crm.segments.contains(vip_segment.id, EntityReference("contact", c1.id)) is True
assert crm.segments.contains(vip_segment.id, EntityReference("contact", c4.id)) is False

print("✨ TOUS LES TESTS DU NOTEBOOK 05 (SEGMENTATION) SONT VALIDÉS AVEC SUCCÈS !")

✨ TOUS LES TESTS DU NOTEBOOK 05 (SEGMENTATION) SONT VALIDÉS AVEC SUCCÈS !
